# Notebook 15 — Latency Benchmark (final verified version)

Verified DeepFM score reproduction using Notebook 10 batch-size consistency check. LightGBM receives a feature-named DataFrame to avoid sklearn warnings. Candidate generation, feature construction, and DataFrame preparation are outside the timed inference region.


## 1. Setup and load frozen September candidate pool

In [1]:
# pip install pandas numpy torch lightgbm scikit-learn gensim scipy
from pathlib import Path
import json, pickle, time, platform, gc
import numpy as np
import pandas as pd
import torch
from torch import nn
ROOT=Path.cwd().resolve()
if not (ROOT/'data'/'processed').is_dir(): ROOT=ROOT.parent
assert (ROOT/'data'/'processed').is_dir(), 'Run from project root or notebooks/'
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; MODELS=ROOT/'models'
REPORTS.mkdir(exist_ok=True)
TOP_N=20; FINAL_K=10; SEED=42
rng=np.random.default_rng(SEED)
torch.set_num_threads(min(4,torch.get_num_threads()))
DEVICE=torch.device('cpu') # consistent single-host CPU comparison
with open(REPORTS/'notebook09_lightgbm_preranker.pkl','rb') as f: lgb_package=pickle.load(f)
lgb=lgb_package['model']; FEATURES=list(lgb_package['features'])
try: package=torch.load(REPORTS/'notebook10_deepfm_model.pt',map_location='cpu',weights_only=True)
except TypeError: package=torch.load(REPORTS/'notebook10_deepfm_model.pt',map_location='cpu')
assert FEATURES==list(package['features']), 'Notebook 09 and 10 feature lists differ'
raw=pd.read_csv(PROC/'notebook08_test_hybrid_candidates.csv')
assert raw.groupby('visitorid').size().eq(100).all()
print('CPU:',platform.processor(),'PyTorch threads:',torch.get_num_threads(),'users:',raw.visitorid.nunique())

CPU: Intel64 Family 6 Model 154 Stepping 3, GenuineIntel PyTorch threads: 4 users: 15137


## 2. Rebuild the **same historical features** as Notebook 10 (outside serving timer)

In [2]:
def make_features(candidates,history,cutoff):
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    stats=views.groupby('itemid',sort=False).agg(item_views=('visitorid','size'),item_unique_viewers=('visitorid','nunique'),item_last_view=('timestamp','max'))
    users=views.groupby('visitorid',sort=False).agg(user_views=('itemid','size'),user_unique_items=('itemid','nunique'),user_last_event=('timestamp','max'))
    other=history.loc[history.event.isin(['addtocart','transaction']),['itemid','event']]
    if len(other):
        counts=other.groupby(['itemid','event']).size().unstack(fill_value=0)
        stats['item_carts']=counts['addtocart'] if 'addtocart' in counts else 0
        stats['item_purchases']=counts['transaction'] if 'transaction' in counts else 0
    else: stats['item_carts']=0;stats['item_purchases']=0
    stats[['item_carts','item_purchases']]=stats[['item_carts','item_purchases']].fillna(0)
    d=candidates.merge(stats,left_on='itemid',right_index=True,how='left',validate='many_to_one')
    d=d.merge(users,left_on='visitorid',right_index=True,how='left',validate='many_to_one')
    ms=int(cutoff.timestamp()*1000)
    d['retrieval_rank']=d['rank'].astype('float32')
    d['rrf_inverse_rank']=(1.0/(60+d['rank'])).astype('float32')
    d['source_count']=d[['from_itemcf','from_faiss_fallback','from_popularity']].sum(axis=1)
    for src,dst in [('item_views','log_item_views'),('item_unique_viewers','log_item_unique_viewers'),('user_views','log_user_views'),('user_unique_items','log_user_unique_items'),('item_carts','log_item_carts'),('item_purchases','log_item_purchases')]:
        d[dst]=np.log1p(d[src].fillna(0)).astype('float32')
    d['user_last_event_days']=((ms-d.user_last_event)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_last_view_days']=((ms-d.item_last_view)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_view_share']=(d.item_views.fillna(0)/max(1,len(views))).astype('float32')
    for name in FEATURES:d[name]=d[name].fillna(0).astype('float32')
    return d[['visitorid','itemid','rank']+FEATURES].copy()
hist=pd.read_csv(PROC/'test_history.csv',usecols=['timestamp','visitorid','itemid','event'])
cutoff=pd.Timestamp('2015-09-01',tz='UTC')
assert hist.timestamp.max()<int(cutoff.timestamp()*1000)
t0=time.perf_counter(); data=make_features(raw,hist,cutoff); feature_build_seconds=time.perf_counter()-t0
assert data[FEATURES].notna().all().all()
print('Historical feature preparation seconds (offline):',round(feature_build_seconds,2))

Historical feature preparation seconds (offline): 2.93


## 3. Restore DeepFM and check score reproduction

In [3]:
class NumericDeepFM(nn.Module):
    def __init__(self,n_fields,embed_dim=8,hidden=(64,32),dropout=.2):
        super().__init__();self.linear=nn.Linear(n_fields,1)
        self.fm_vectors=nn.Parameter(torch.randn(n_fields,embed_dim)*.02)
        layers=[];d=n_fields*embed_dim
        for width in hidden:
            layers.extend([nn.Linear(d,width),nn.ReLU(),nn.Dropout(dropout)]);d=width
        layers.append(nn.Linear(d,1));self.mlp=nn.Sequential(*layers)
    def forward(self,x):
        e=x.unsqueeze(-1)*self.fm_vectors.unsqueeze(0);s=e.sum(dim=1)
        fm=.5*(s.square()-(e.square()).sum(dim=1)).sum(dim=1,keepdim=True)
        return (self.linear(x)+fm+self.mlp(e.flatten(start_dim=1))).squeeze(-1)
deepfm=NumericDeepFM(len(FEATURES),package['embed_dim'],package['hidden'],package['dropout']).to(DEVICE)
deepfm.load_state_dict(package['state_dict']);deepfm.eval()
mean=np.asarray(package['scaler_mean'],dtype=np.float32);scale=np.asarray(package['scaler_scale'],dtype=np.float32)
def deep_predict(x):
    z=np.clip((np.asarray(x,dtype=np.float32)-mean)/scale,-8,8)
    with torch.inference_mode():return deepfm(torch.from_numpy(z)).numpy()
# Validate a small sample against exported Notebook 10 scores (float tolerances due to CPU/library).
reference=pd.read_csv(PROC/'notebook10_test_scored_candidates.csv',usecols=['visitorid','itemid','score'])
probe=data.head(300).copy();actual=deep_predict(probe[FEATURES].to_numpy(dtype=np.float32))
check=probe[['visitorid','itemid']].copy();check['pred']=actual
check=check.merge(reference,on=['visitorid','itemid'],validate='one_to_one')
max_err=float(np.max(np.abs(check.pred-check.score)))
# Diagnose model / feature discrepancies without silently accepting invalid replay.
check['abs_error']=(check['pred']-check['score']).abs()
check=check.sort_values('abs_error',ascending=False)
print('Score reproduction: max=',max_err,' median=',float(check.abs_error.median()),
      ' p95=',float(check.abs_error.quantile(.95)))
print('Largest mismatches:')
print(check.head(10).to_string(index=False))
print('Saved model metadata:',{k:package.get(k) for k in ('features','best_epoch','scaler_mean','scaler_scale') if k in package})
print('Feature ranges for mismatched rows:')
print(probe.merge(check.head(5)[['visitorid','itemid']],on=['visitorid','itemid'])[FEATURES].to_string(index=False))
# Cross-check the same model on identical inputs with Notebook10's 16,384-row batch.
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.mean_=np.asarray(package['scaler_mean'],dtype=np.float64)
scaler.scale_=np.asarray(package['scaler_scale'],dtype=np.float64)
scaler.var_=scaler.scale_**2
scaler.n_features_in_=len(FEATURES)
probe_x=np.clip(scaler.transform(probe[FEATURES].to_numpy(dtype=np.float32)),-8,8).astype('float32')
with torch.no_grad():
    pred_batch300=deepfm(torch.from_numpy(probe_x)).cpu().numpy()
# Notebook10 scored batches of 16384; probe itself is only 300 rows, so compare
# inference using 16384 rows from the same snapshot and extract the first 300.
full_x=np.clip(scaler.transform(data.iloc[:16384][FEATURES].to_numpy(dtype=np.float32)),-8,8).astype('float32')
with torch.no_grad():
    pred_batch16384=deepfm(torch.from_numpy(full_x)).cpu().numpy()[:len(probe)]
size_err=float(np.max(np.abs(pred_batch300-pred_batch16384)))
ref_err=float(np.max(np.abs(pred_batch16384-check.set_index(['visitorid','itemid']).loc[
    list(zip(probe.visitorid,probe.itemid)),'score'].to_numpy())))
print('Batch-size numerical difference:',size_err)
print('Saved CSV vs Notebook10-sized inference:',ref_err)
if ref_err >= 1e-3:
    raise AssertionError('Saved model and exported scores do not reproduce. '
       'Compare batch-size difference and CSV difference above; verify both files '
       'were generated in the same uninterrupted Notebook10 run. '
       'Do not benchmark against a mismatched model artifact.')
print('PASS: Saved DeepFM model reproduces exported test scores')


Score reproduction: max= 1.184509277329937e-06  median= 1.073813819946201e-07  p95= 4.6168350225084255e-07
Largest mismatches:
 visitorid  itemid      pred     score    abs_error
       278  249688 -4.339822 -4.339823 1.184509e-06
       278  336501 -3.697157 -3.697157 7.322906e-07
       278  106697 -3.206037 -3.206037 6.829437e-07
       278  279940 -3.329134 -3.329135 5.357361e-07
       278   91290 -4.687362 -4.687361 5.172241e-07
       278  349973 -4.009606 -4.009606 5.154480e-07
       278  432435 -3.368528 -3.368528 5.107483e-07
       278   38793 -3.748343 -3.748343 5.091248e-07
       278  348339 -3.801346 -3.801347 4.979675e-07
       155   50261 -2.485252 -2.485252 4.964661e-07
Saved model metadata: {'features': ['retrieval_rank', 'rrf_inverse_rank', 'from_itemcf', 'from_faiss_fallback', 'from_popularity', 'source_count', 'log_item_views', 'log_item_unique_viewers', 'log_user_views', 'log_user_unique_items', 'user_last_event_days', 'item_last_view_days', 'item_view_share', 

## 4. Per-user conditional inference benchmark (CPU, warmed up)

In [4]:
# Pre-materialized feature matrices mimic an online feature store; exclude dataframe extraction and I/O.
user_ids=np.sort(data.visitorid.unique())
SAMPLE_USERS=min(1000,len(user_ids));chosen=rng.choice(user_ids,size=SAMPLE_USERS,replace=False)
by_user={int(u):grp for u,grp in data.loc[data.visitorid.isin(chosen)].groupby('visitorid',sort=False)}
prepared={u:(g['itemid'].to_numpy(),g[FEATURES].to_numpy(dtype=np.float32),g[FEATURES].copy()) for u,g in by_user.items()}
# Keep named feature DataFrames pre-materialized so benchmark measures model inference,
# not DataFrame creation. The same input feature order is used for both models.
assert list(next(iter(prepared.values()))[2].columns)==FEATURES
def run_user(items,x,x_named,gate=20):
    a=time.perf_counter_ns();ls=lgb.predict(x_named);b=time.perf_counter_ns()
    # stable tie-break by item ID
    idx=np.lexsort((items,-ls))[:gate]
    c=time.perf_counter_ns();ds=deep_predict(x[idx]);d=time.perf_counter_ns()
    order=np.lexsort((items[idx],-ds))[:FINAL_K]
    result=items[idx][order];e=time.perf_counter_ns()
    return result,{'lightgbm_ms':(b-a)/1e6,'gate_ms':(c-b)/1e6,'deepfm_ms':(d-c)/1e6,'sort_ms':(e-d)/1e6,'total_ms':(e-a)/1e6}
for u in list(prepared)[:30]:
    items,x,x_named=prepared[u];run_user(items,x,x_named,20);run_user(items,x,x_named,100)
gc.collect()
records=[];outputs={}
for u,(items,x,x_named) in prepared.items():
    for gate in (20,100):
        result,t=run_user(items,x,x_named,gate)
        outputs[(u,gate)]=result
        records.append({'visitorid':u,'gate':gate,**t})
lat=pd.DataFrame(records)
def aggregate(frame):
    row={'users':len(frame)}
    for name in ('lightgbm_ms','gate_ms','deepfm_ms','sort_ms','total_ms'):
        a=frame[name].to_numpy();row[name+'_p50']=float(np.percentile(a,50));row[name+'_p95']=float(np.percentile(a,95));row[name+'_mean']=float(a.mean())
    return row
metrics={str(g):aggregate(lat.loc[lat.gate.eq(g)]) for g in (20,100)}
print(pd.DataFrame(metrics).T[['total_ms_p50','total_ms_p95','deepfm_ms_p50','deepfm_ms_p95']])

     total_ms_p50  total_ms_p95  deepfm_ms_p50  deepfm_ms_p95
20        2.91125       4.10809        0.60255        0.98226
100       3.03040       4.31878        0.73735        1.13914


## 5. Replay agreement with Notebook 13 (without MMR)

The reference output uses the same frozen LightGBM gate and DeepFM model. The benchmark excludes MMR and retrieval. Different numerical precision or score ties may cause small differences; we report rather than hide them.

In [5]:
# Compare selected Top-20 outputs with Notebook 13 gated DeepFM ranking where possible.
# notebook13 final Top-10 is after MMR; compare candidate sets only when raw stage output is available.
# Verify gate really limits DeepFM to 20, and p95 is computed over per-user measurements.
assert len(lat)==2*SAMPLE_USERS
assert all(len(v)==10 for v in outputs.values())
assert all(len(set(map(int,v)))==10 for v in outputs.values())
# Note: deepfm p95 is not guaranteed to improve proportionally due to fixed per-call overhead.
paired=lat.pivot(index='visitorid',columns='gate',values='total_ms')
paired['delta_ms_100_minus_20']=paired[100]-paired[20]
print('Mean total latency saving (ms):',round(float(paired.delta_ms_100_minus_20.mean()),4))
print('Fraction users with faster Top20:',round(float((paired.delta_ms_100_minus_20>0).mean()),4))

Mean total latency saving (ms): 0.1024
Fraction users with faster Top20: 0.536


## 6. Export reproducible benchmark report

In [6]:
report={'method':'local CPU per-user conditional LightGBM-to-DeepFM microbenchmark',
 'protocol':'September cutoff; frozen model artifacts; precomputed candidate and historical features',
 'sampled_users':int(SAMPLE_USERS),'top_n':TOP_N,'candidate_pool':100,
 'model_reproduction_max_abs_error':max_err,'offline_feature_build_seconds':float(feature_build_seconds),
 'latency_ms':metrics,
 'paired_mean_total_saving_ms':float(paired.delta_ms_100_minus_20.mean()),
 'fraction_users_top20_faster':float((paired.delta_ms_100_minus_20>0).mean()),
 'hardware':{'platform':platform.platform(),'processor':platform.processor(),'torch_threads':torch.get_num_threads()},
 'excluded':['retrieval generation','feature store lookup / historical feature computation','model loading','serialization','network','MMR diversity re-ranking'],
 'caveats':['This is local CPU microbenchmark, not production p95','One-user-at-a-time CPU inference, no concurrency or batching across users','All 100 candidates receive LightGBM; only 20 receive DeepFM in gated run','Top20 and Top100 measured sequentially; system load affects timing','RetailRocket has no ad impressions or auction data']}
out=REPORTS/'notebook15_latency_benchmark.json'
out.write_text(json.dumps(report,indent=2),encoding='utf-8')
lat.to_csv(REPORTS/'notebook15_per_user_latency.csv',index=False)
print('Saved:',out)

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook15_latency_benchmark.json


## Interpretation

Compare **measured** Top-20 vs Top-100 DeepFM inference and end-to-end **ranking-only** latency. Do not claim 80% latency improvement based solely on reducing candidate count. Next: Notebook 16 — FastAPI local serving demo.